In [1]:
import sys, pathlib
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.dataraflow_db import run_query

london = run_query("SELECT * FROM week07.london_2014")
london.head()

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees<br />
0,2014-1-1,11,8,6,9,7,4,94,86,73,...,31,11,2,40,26,66.0,9.91,4.0,Rain,186<br />
1,2014-1-2,11,9,6,9,6,4,94,81,60,...,31,15,5,32,21,50.0,1.02,2.0,Rain,214<br />
2,2014-1-3,11,8,6,9,5,2,94,76,54,...,31,12,0,50,29,69.0,7.11,2.0,Rain-Thunderstorm,219<br />
3,2014-1-4,9,6,3,8,6,2,93,85,65,...,31,12,5,35,21,39.0,9.91,4.0,Rain,211<br />
4,2014-1-5,12,7,1,11,4,-1,100,88,77,...,31,14,5,35,16,NaN,0.25,5.0,Rain,199<br />


In [4]:
# Filter / aggregate example

warm_days = run_query("""
    SELECT GMT, [Max TemperatureC]
    FROM week07.london_2014
    WHERE [Max TemperatureC] >= 25
    ORDER BY [Max TemperatureC] DESC
""")

london.to_csv("london_2014_clean.csv", index=False)

In [5]:
# Option 1: JOIN in SQL (recommended this week)
#Let the database do the merge and hand you one tidy table:

combined = run_query("""
    SELECT g.country,
           g.[NY.GDP.MKTP.CD] AS gdp,
           l.[SP.DYN.LE00.IN] AS life_expectancy,
           p.[SP.POP.TOTL]    AS population
    FROM week08.wb_gdp_2013 g
    JOIN week08.wb_le_2013  l ON g.country = l.country
    JOIN week08.wb_pop_2013 p ON g.country = p.country
""")
combined.shape   # (248, 4)

(248, 4)

In [6]:
# Option 2: pull separately, merge in pandas


gdp  = run_query("SELECT country, [NY.GDP.MKTP.CD] AS gdp FROM week08.wb_gdp_2013")
life = run_query("SELECT country, [SP.DYN.LE00.IN] AS life_expectancy FROM week08.wb_le_2013")
pop  = run_query("SELECT country, [SP.POP.TOTL] AS population FROM week08.wb_pop_2013")

combined = gdp.merge(life, on="country").merge(pop, on="country")

In [7]:
# Average life expectancy for the 20 largest economies
top_life = run_query("""
    SELECT TOP 20 g.country,
           g.[NY.GDP.MKTP.CD] AS gdp,
           l.[SP.DYN.LE00.IN] AS life_expectancy
    FROM week08.wb_gdp_2013 g
    JOIN week08.wb_le_2013 l ON g.country = l.country
    ORDER BY g.[NY.GDP.MKTP.CD] DESC
""")
print(top_life["life_expectancy"].mean())

combined.to_csv("wb_combined_2013.csv", index=False)

75.92023866299134


In [9]:
import warnings
warnings.simplefilter('ignore', FutureWarning)

import pandas as pd
from datetime import datetime

In [10]:
# Load Data from cohort database (schema week07). See "Data Access (SQL).md".
import sys, pathlib
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.dataraflow_db import run_query

london = run_query("SELECT * FROM week07.london_2014")
london.head()

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees<br />
0,2014-1-1,11,8,6,9,7,4,94,86,73,...,31,11,2,40,26,66.0,9.91,4.0,Rain,186<br />
1,2014-1-2,11,9,6,9,6,4,94,81,60,...,31,15,5,32,21,50.0,1.02,2.0,Rain,214<br />
2,2014-1-3,11,8,6,9,5,2,94,76,54,...,31,12,0,50,29,69.0,7.11,2.0,Rain-Thunderstorm,219<br />
3,2014-1-4,9,6,3,8,6,2,93,85,65,...,31,12,5,35,21,39.0,9.91,4.0,Rain,211<br />
4,2014-1-5,12,7,1,11,4,-1,100,88,77,...,31,14,5,35,16,NaN,0.25,5.0,Rain,199<br />


In [11]:
london.columns

Index(['GMT', 'Max TemperatureC', 'Mean TemperatureC', 'Min TemperatureC',
       'Dew PointC', 'MeanDew PointC', 'Min DewpointC', 'Max Humidity',
       ' Mean Humidity', ' Min Humidity', ' Max Sea Level PressurehPa',
       ' Mean Sea Level PressurehPa', ' Min Sea Level PressurehPa',
       ' Max VisibilityKm', ' Mean VisibilityKm', ' Min VisibilitykM',
       ' Max Wind SpeedKm/h', ' Mean Wind SpeedKm/h', ' Max Gust SpeedKm/h',
       'Precipitationmm', ' CloudCover', ' Events', 'WindDirDegrees<br />'],
      dtype='str')

In [13]:
# Previously this reloaded the CSV with read_csv(..., skipinitialspace=True) to drop the
# spaces after commas. Pulling from SQL, those spaces are already part of the column
# names, so we strip them directly in pandas -- same result:
london = run_query("SELECT * FROM week07.london_2014")
london.columns = london.columns.str.strip()
london.head()

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees<br />
0,2014-1-1,11,8,6,9,7,4,94,86,73,...,31,11,2,40,26,66.0,9.91,4.0,Rain,186<br />
1,2014-1-2,11,9,6,9,6,4,94,81,60,...,31,15,5,32,21,50.0,1.02,2.0,Rain,214<br />
2,2014-1-3,11,8,6,9,5,2,94,76,54,...,31,12,0,50,29,69.0,7.11,2.0,Rain-Thunderstorm,219<br />
3,2014-1-4,9,6,3,8,6,2,93,85,65,...,31,12,5,35,21,39.0,9.91,4.0,Rain,211<br />
4,2014-1-5,12,7,1,11,4,-1,100,88,77,...,31,14,5,35,16,NaN,0.25,5.0,Rain,199<br />


In [14]:
london['WindDirDegrees<br />'].head()

0    186<br />
1    214<br />
2    219<br />
3    211<br />
4    199<br />
Name: WindDirDegrees<br />, dtype: str

In [15]:
london = london.rename(columns={'WindDirDegrees<br />' : 'WindDirDegrees'})

In [16]:
london['WindDirDegrees'] = london['WindDirDegrees'].str.rstrip('<br />')

In [17]:
london['WindDirDegrees'].head()

0    186
1    214
2    219
3    211
4    199
Name: WindDirDegrees, dtype: str

In [18]:
london[london['Events'].isnull()]

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees
32,2014-2-2,10,8,5,4,3,1,93,71,43,...,31,16,10,37,24,47.0,0.00,2.0,NaN,216
33,2014-2-3,8,6,3,5,3,2,93,79,55,...,31,15,10,32,18,NaN,0.00,4.0,NaN,133
43,2014-2-13,8,6,3,2,1,-1,81,66,45,...,31,15,10,42,27,48.0,0.00,2.0,NaN,236
52,2014-2-22,12,8,4,5,3,1,87,68,38,...,31,16,10,29,21,NaN,0.00,2.0,NaN,228
63,2014-3-5,13,7,0,4,2,-1,100,72,40,...,31,15,8,21,6,NaN,0.25,1.0,NaN,248
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351,2014-12-18,13,12,11,11,11,10,94,85,74,...,31,12,10,37,29,60.0,0.00,6.0,NaN,245
358,2014-12-25,8,6,2,3,2,1,93,79,51,...,31,19,10,24,16,NaN,0.00,1.0,NaN,275
361,2014-12-28,6,2,-2,1,-2,-3,100,76,45,...,31,17,8,16,11,NaN,0.00,1.0,NaN,5
362,2014-12-29,6,1,-4,2,-1,-4,100,87,62,...,24,10,5,13,8,NaN,0.25,1.0,NaN,289


In [19]:
london[london['Events'].isnull()]

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees
32,2014-2-2,10,8,5,4,3,1,93,71,43,...,31,16,10,37,24,47.0,0.00,2.0,NaN,216
33,2014-2-3,8,6,3,5,3,2,93,79,55,...,31,15,10,32,18,NaN,0.00,4.0,NaN,133
43,2014-2-13,8,6,3,2,1,-1,81,66,45,...,31,15,10,42,27,48.0,0.00,2.0,NaN,236
52,2014-2-22,12,8,4,5,3,1,87,68,38,...,31,16,10,29,21,NaN,0.00,2.0,NaN,228
63,2014-3-5,13,7,0,4,2,-1,100,72,40,...,31,15,8,21,6,NaN,0.25,1.0,NaN,248
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351,2014-12-18,13,12,11,11,11,10,94,85,74,...,31,12,10,37,29,60.0,0.00,6.0,NaN,245
358,2014-12-25,8,6,2,3,2,1,93,79,51,...,31,19,10,24,16,NaN,0.00,1.0,NaN,275
361,2014-12-28,6,2,-2,1,-2,-3,100,76,45,...,31,17,8,16,11,NaN,0.00,1.0,NaN,5
362,2014-12-29,6,1,-4,2,-1,-4,100,87,62,...,24,10,5,13,8,NaN,0.25,1.0,NaN,289


In [20]:
london.dropna()

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees
0,2014-1-1,11,8,6,9,7,4,94,86,73,...,31,11,2,40,26,66.0,9.91,4.0,Rain,186
1,2014-1-2,11,9,6,9,6,4,94,81,60,...,31,15,5,32,21,50.0,1.02,2.0,Rain,214
2,2014-1-3,11,8,6,9,5,2,94,76,54,...,31,12,0,50,29,69.0,7.11,2.0,Rain-Thunderstorm,219
3,2014-1-4,9,6,3,8,6,2,93,85,65,...,31,12,5,35,21,39.0,9.91,4.0,Rain,211
5,2014-1-6,12,10,8,11,8,6,94,78,60,...,31,12,5,50,29,61.0,2.03,4.0,Rain,219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,2014-12-22,13,12,10,11,9,7,94,77,63,...,31,14,8,40,32,55.0,0.00,5.0,Rain,240
356,2014-12-23,13,11,10,9,7,5,82,72,58,...,31,12,10,42,32,61.0,0.00,4.0,Rain,238
357,2014-12-24,12,8,4,10,4,-2,94,71,38,...,31,13,10,34,26,45.0,0.25,5.0,Rain,262
360,2014-12-27,4,3,2,8,1,-1,93,83,64,...,31,13,9,42,26,61.0,0.00,4.0,Rain,330


In [21]:
london.dtypes

GMT                               str
Max TemperatureC                int64
Mean TemperatureC               int64
Min TemperatureC                int64
Dew PointC                      int64
MeanDew PointC                  int64
Min DewpointC                   int64
Max Humidity                    int64
Mean Humidity                   int64
Min Humidity                    int64
Max Sea Level PressurehPa       int64
Mean Sea Level PressurehPa      int64
Min Sea Level PressurehPa       int64
Max VisibilityKm                int64
Mean VisibilityKm               int64
Min VisibilitykM                int64
Max Wind SpeedKm/h              int64
Mean Wind SpeedKm/h             int64
Max Gust SpeedKm/h            float64
Precipitationmm               float64
CloudCover                    float64
Events                            str
WindDirDegrees                    str
dtype: object

In [22]:
london['WindDirDegrees'] = london['WindDirDegrees'].astype('int64')  

In [23]:
london['GMT'] = pd.to_datetime(london['GMT'])
london.dtypes

GMT                           datetime64[us]
Max TemperatureC                       int64
Mean TemperatureC                      int64
Min TemperatureC                       int64
Dew PointC                             int64
MeanDew PointC                         int64
Min DewpointC                          int64
Max Humidity                           int64
Mean Humidity                          int64
Min Humidity                           int64
Max Sea Level PressurehPa              int64
Mean Sea Level PressurehPa             int64
Min Sea Level PressurehPa              int64
Max VisibilityKm                       int64
Mean VisibilityKm                      int64
Min VisibilitykM                       int64
Max Wind SpeedKm/h                     int64
Mean Wind SpeedKm/h                    int64
Max Gust SpeedKm/h                   float64
Precipitationmm                      float64
CloudCover                           float64
Events                                   str
WindDirDeg

In [24]:
london[london['GMT'] == datetime(2014, 6, 4)] 

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees
154,2014-06-04,14,11,8,11,9,3,94,78,47,...,31,12,7,47,13,64.0,2.03,4.0,Rain-Thunderstorm,230


In [ ]:
#Task 1
# write code to select all days that had a northerly wind. Hint: select the rows where the direction is greater than or equal to 350 or smaller than or equal to 10, as the compass rose shows.

northerly = london[(london['WindDirDegrees'] >= 350) | (london['WindDirDegrees'] <= 10)]
northerly

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees
84,2014-03-26,9,6,2,4,2,-1,100,80,39,...,27,10,1,26,10,NaN,2.03,4.0,Fog-Rain,3
90,2014-04-01,19,13,7,11,9,6,100,80,39,...,19,7,0,16,5,NaN,0.00,4.0,Fog-Rain,6
100,2014-04-11,16,12,9,5,4,2,71,53,28,...,31,18,10,13,8,NaN,0.00,1.0,NaN,351
110,2014-04-21,19,12,5,11,8,4,100,78,31,...,26,7,0,14,6,NaN,9.91,3.0,Fog-Rain-Thunderstorm,356
174,2014-06-24,24,19,13,15,12,9,94,66,27,...,31,12,0,23,8,37.0,0.00,3.0,Fog,350
179,2014-06-29,19,16,12,11,9,5,88,65,31,...,31,16,10,19,10,NaN,0.00,2.0,Rain,9
190,2014-07-10,22,18,14,13,11,9,82,64,44,...,31,19,10,16,14,NaN,0.00,3.0,NaN,1
208,2014-07-28,23,19,14,16,12,9,94,66,33,...,26,12,2,21,10,NaN,17.02,2.0,Rain-Thunderstorm,3
249,2014-09-07,21,17,13,14,12,8,94,71,41,...,23,10,2,11,6,NaN,0.00,4.0,NaN,354
250,2014-09-08,22,16,10,13,9,7,94,64,28,...,27,12,5,11,6,NaN,0.00,1.0,NaN,351


In [ ]:
# Task 2
# write code to get and display all the rows in the dataframe that are beween 1 April 2014 and 11 April 2014.

import datetime

london['GMT'] = pd.to_datetime(london['GMT'])

april_rows = london[(london['GMT'] >= '2014-04-01') & (london['GMT'] <= '2014-04-11')]
april_rows

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees
90,2014-04-01,19,13,7,11,9,6,100,80,39,...,19,7,0,16,5,NaN,0.00,4.0,Fog-Rain,6
91,2014-04-02,18,14,11,11,9,7,88,71,48,...,10,8,5,23,13,NaN,0.00,1.0,Rain,85
92,2014-04-03,18,14,11,11,9,9,88,75,52,...,21,7,3,26,11,NaN,0.00,5.0,NaN,96
93,2014-04-04,15,12,9,10,8,6,88,71,46,...,31,11,2,24,14,NaN,0.00,5.0,NaN,253
94,2014-04-05,14,10,6,11,8,5,100,83,56,...,31,12,1,26,13,50.0,0.00,5.0,Fog-Rain,197
95,2014-04-06,16,14,12,12,12,11,94,88,67,...,31,12,5,35,21,48.0,2.03,6.0,Rain,213
96,2014-04-07,14,11,8,12,11,6,94,87,76,...,31,14,5,29,23,42.0,2.03,5.0,Rain,209
97,2014-04-08,14,9,5,6,3,-1,87,64,26,...,31,15,10,39,23,NaN,0.00,2.0,Rain,269
98,2014-04-09,17,10,4,7,6,3,93,68,38,...,31,14,6,23,13,NaN,0.00,2.0,NaN,261
99,2014-04-10,16,11,6,6,5,3,93,62,31,...,31,14,9,19,10,NaN,0.00,3.0,NaN,278


In [ ]:
# Task 3
# write two lines of code to display the first five rows that have a missing value in the 'Max Gust SpeedKm/h' column

missing_gust = london[london['Max Gust SpeedKm/h'].isnull()]
missing_gust.head()

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees
4,2014-01-05,12,7,1,11,4,-1,100,88,77,...,31,14,5,35,16,NaN,0.25,5.0,Rain,199
10,2014-01-11,9,4,0,7,3,-1,100,80,51,...,31,12,6,24,14,NaN,0.00,4.0,Rain,259
11,2014-01-12,10,4,-2,9,3,-2,100,91,79,...,31,9,1,32,10,NaN,0.51,6.0,Fog-Rain,126
12,2014-01-13,9,7,5,8,4,3,93,85,71,...,31,16,5,34,19,NaN,5.08,2.0,Rain,205
13,2014-01-14,7,4,1,5,2,0,100,82,55,...,31,15,10,21,13,NaN,0.25,3.0,Rain,242
